# Modern Python Features 3.10 To 3.13

## Quick notes

### What this covers
The language and library features added from Python 3.10 to 3.13, each demonstrated behind a version or feature check so the file runs on 3.10+ and prints a skip line for anything newer than the interpreter. Syntax older Pythons cannot parse is kept in strings and run with `exec()` only when the version allows.

### Key points
- **Version guards**: compare tuples (`sys.version_info >= (3, 12)`), never strings. Use the version check for syntax/behaviour, `hasattr` or `try: import ... except ImportError` for APIs that may have a backport (`typing_extensions`, `tomli`).
- **`match`/`case` (3.10)** compares shape and destructures: literals, captures, `_`, or-patterns `A | B`, `as` names, guards `if cond`, class patterns `Point(x=0)` or positional (needs `__match_args__`; dataclasses provide it), sequence `[a, *rest]`, mapping `{"k": v, **rest}`. Constants must be dotted names (`Color.RED`).
- **Parenthesized `with` (3.10)**: several context managers over several lines, trailing comma allowed.
- **`ExceptionGroup` / `except*` (3.11)** handle many simultaneous errors; `split`, `subgroup`; `add_note()` fills `__notes__`.
- **PEP 695 (3.12)**: `def f[T](x: T)`, `class Box[T]`, bounds `[T: int]`, constraints `[T: (int, str)]`, and `type Alias[T] = ...` (lazy, so recursive aliases need no quotes).
- **Typing additions**: `TypeGuard` (3.10), `Self`, `LiteralString`, `Required`/`NotRequired`, `Unpack` (3.11), `override` (3.12), `TypeIs` (3.13). All are hints; nothing is enforced at run time.
- **f-strings (3.12)**: same quotes inside, backslashes, comments and multi-line expressions, nesting.
- **`tomllib` (3.11)** reads TOML (`load` needs a binary file; no writer).
- **`itertools.pairwise` (3.10)**, **`batched` (3.12)**, `strict=True` added in 3.13.
- **Digit limit (3.11, backported 3.10.7)**: decimal `int`/`str` conversions above 4300 digits raise `ValueError`; `sys.set_int_max_str_digits(n)`, `0` disables; hex/binary are not limited.
- **Error messages**: "Did you mean" (3.10), exact-position carets (3.11), better import and attribute hints (3.12), colour (3.13).
- **`sys.monitoring` (3.12)** low-overhead event hooks for tools: claim a tool id, register callbacks, `set_events`, free the id.
- **3.13**: free-threaded build (`python3.13t`, detect with `sys._is_gil_enabled()` and `sysconfig` `Py_GIL_DISABLED`), experimental JIT (build option), new REPL (multi-line editing, F1/F2/F3, plain `exit`).
- **Removals**: 3.12 dropped `distutils`, `asyncore`, `asynchat`, `smtpd`, `imp`; 3.13 dropped the PEP 594 "dead batteries" (`cgi`, `telnetlib`, `crypt`, `pipes`, `imghdr`, `uu`, `xdrlib` and more).

### Gotchas (the part worth revising twice)
- **A bare name in `case` captures; it does not compare.** `case RED:` matches everything and rebinds `RED`. Use `Color.RED` or a literal.
- **`str` is not matched by sequence patterns**, but `True`/`1.0` do match `case 1` (equality). Order cases or use `int()` carefully.
- **Variables bound in a `case` outlive the match.** `match`, `case` and `type` are soft keywords and stay valid variable names.
- **An or-pattern must bind the same names in every branch.**
- **`except*` cannot be mixed with `except`**, and cannot contain `return`/`break`/`continue`; empty groups are rejected.
- **New syntax breaks older Pythons at parse time**, so a version `if` is not enough: 3.12 f-strings or `def f[T]` in a file still supporting 3.11 are a `SyntaxError`.
- **Type hints are not checked at run time**; a `TypeGuard` that lies silently breaks the checker's trust. `TypeIs` must be a true narrowing test.
- **"Did you mean" lives in the traceback text, not in `str(err)`.** Do not parse it.
- **The digit limit is a security feature**; do not set it to `0` for untrusted input. Restore it after temporary changes.
- **Free-threading** is experimental in 3.13: slower single thread, extensions must be rebuilt, and the GIL used to hide races. Do not assume your data is thread-safe.
- **`sys._is_gil_enabled` is private and missing before 3.13**; guard with `hasattr`.
- **Compare versions as tuples**: `"3.9" > "3.12"` is `True` as strings.

### When it matters
Writing libraries that support several Python versions, upgrading old code bases (finding removed modules early with `-W error::DeprecationWarning`), parsing structured data with `match`, handling concurrent failures, and deciding whether a free-threaded build or JIT is worth trying.

### One-line recap example
```python
import sys
if sys.version_info >= (3, 10):
    match {"type": "click", "pos": (1, 2)}:
        case {"type": "click", "pos": (x, y)}: print(x, y)   # 1 2
```

In [1]:
import importlib.util
import itertools
import sys
import textwrap
import traceback

## 01. Index And The Version-Guard Pattern

- 01 Index + version guards          12 Typing: Self, LiteralString, Unpack
- 02 Structural matching: basics     13 Typing: TypeGuard and TypeIs
- 03 Matching: class patterns        14 f-string relaxations (3.12)
- 04 Matching: mapping and sequence  15 tomllib (3.11)
- 05 Matching: guards, or, as        16 itertools: pairwise and batched
- 06 Matching gotchas                17 int <-> str digit limit
- 07 Parenthesized context managers  18 Better error messages
- 08 ExceptionGroup and except*      19 sys.monitoring (3.12)
- 09 add_note and group methods      20 3.13: free-threading, JIT, new REPL
- 10 PEP 695: def f[T], class C[T]   21 Deprecations and removals
- 11 PEP 695: type X = ...           22 Recap
- Every feature below is gated by a version check. Syntax that older Pythons
  cannot even PARSE is kept inside strings and run with exec() only when the
  version is high enough. That way this file still runs (and exits 0) on 3.10
  and just prints a skip line for features it does not have. On Python older
  than 3.10 the whole file prints one line and stops.
- Version guard cheat sheet:
    sys.version_info >= (3, 12)       compare a TUPLE, never a string
    hasattr(module, "name")           feature detection: best for library APIs
    try: import tomllib except ImportError: ...   fallback to a backport
  Use the version check for SYNTAX and behaviour changes, hasattr/try-import
  for functions that may exist in a backport (typing_extensions, tomli).

In [2]:
if sys.version_info < (3, 10):
    print("Python 3.10+ is required for this lesson; skipping everything.")
    sys.exit(0)

PY = sys.version_info[:2]
ns = {"__name__": "__main__", "traceback": traceback}  # shared namespace for exec()'d snippets


def run(min_version, code):
    # Run a code snippet only if this Python is new enough; otherwise say so.
    if sys.version_info >= min_version:
        exec(textwrap.dedent(code), ns)
    else:
        print(f"  skipped: needs Python {min_version[0]}.{min_version[1]}+")


print("version check is a tuple compare:", PY >= (3, 10), "| version_info has major/minor:", sys.version_info.major >= 3)
print("comparing strings is wrong ('3.9' > '3.12'):", "3.9" > "3.12", "| tuples are right:", (3, 9) > (3, 12))

version check is a tuple compare: True | version_info has major/minor: True
comparing strings is wrong ('3.9' > '3.12'): True | tuples are right: False


## 02. Structural Pattern Matching - Basics (3.10)

- match/case is NOT a C-style switch. It compares the SHAPE of a value and
  can pull pieces out of it ("destructuring") in the same step.
- Patterns tried top to bottom; the first match runs; no fall-through.
- Literal patterns (1, "a", None, True) compare by equality (None/True/False
  by identity). A bare NAME is a CAPTURE: it matches anything and binds it.
- "_" is the wildcard: matches anything, binds nothing.

In [3]:
run((3, 10), '''
def http_text(code):
    match code:
        case 200:
            return "ok"
        case 301 | 302:          # or-pattern
            return "redirect"
        case 404:
            return "not found"
        case int(n) if n >= 500:  # class pattern + guard
            return f"server error {n}"
        case _:
            return "other"

print("basics:", [http_text(c) for c in (200, 302, 404, 503, 418)])
print("no match and no wildcard simply does nothing (no error)")
''')

basics: ['ok', 'redirect', 'not found', 'server error 503', 'other']
no match and no wildcard simply does nothing (no error)


## 03. Matching - Class Patterns

- case Point(x=0, y=0): checks isinstance(value, Point) and then the named
  attributes. Keyword form works for any class.
- Positional form case Point(0, 0) needs __match_args__ = ("x", "y");
  dataclasses and namedtuples create it for you.
- Built-ins int(), str(), float(), list() ... work as type checks.

In [4]:
run((3, 10), '''
from dataclasses import dataclass

@dataclass
class Point:
    x: int
    y: int

@dataclass
class Circle:
    centre: Point
    radius: int

def describe(shape):
    match shape:
        case Point(x=0, y=0):
            return "origin"
        case Point(x, 0):               # positional, uses __match_args__
            return f"on x axis at {x}"
        case Point(x, y):
            return f"point {x},{y}"
        case Circle(Point(0, 0), r):    # nested pattern
            return f"circle at origin r={r}"
        case Circle(centre=c, radius=r):
            return f"circle at {c.x},{c.y} r={r}"
        case str(s):
            return f"text {s!r}"
        case _:
            return "unknown"

for s in (Point(0, 0), Point(5, 0), Point(1, 2), Circle(Point(0, 0), 3), Circle(Point(1, 1), 2), "hi", 3.5):
    print(" ", describe(s))
print("__match_args__ made by dataclass:", Point.__match_args__)
''')

  origin
  on x axis at 5
  point 1,2
  circle at origin r=3
  circle at 1,1 r=2
  text 'hi'
  unknown
__match_args__ made by dataclass: ('x', 'y')


## 04. Matching - Mapping And Sequence Patterns

- Sequence pattern [a, b, *rest] matches lists and tuples (any Sequence)
  but NOT str/bytes/bytearray, and not dicts or sets.
- *rest (or *_) is a star capture; at most one per sequence pattern.
- Mapping pattern {"key": value, **rest} matches any Mapping that HAS those
  keys; extra keys are ignored unless you capture them with **rest.
- Real use: handling parsed JSON, command lines, tokens, AST nodes.

In [5]:
run((3, 10), '''
def command(parts):
    match parts:
        case []:
            return "empty"
        case ["quit"]:
            return "bye"
        case ["go", ("north" | "south") as direction]:
            return f"walk {direction}"
        case ["add", *numbers] if numbers:
            return f"sum {sum(int(n) for n in numbers)}"
        case [first, *_, last]:
            return f"first={first} last={last}"
        case _:
            return "?"

for c in ([], ["quit"], ["go", "north"], ["add", "1", "2", "3"], ["a", "b", "c"], ["go", "east"], ("quit",), "quit"):
    print(" ", repr(c), "->", command(c))

def event(e):
    match e:
        case {"type": "click", "pos": (x, y)}:
            return f"click at {x},{y}"
        case {"type": "key", "key": k, **rest}:
            return f"key {k} extras={sorted(rest)}"
        case {"type": t}:
            return f"unhandled {t}"
        case _:
            return "not an event"

print(" ", event({"type": "click", "pos": (3, 4), "extra": 1}))
print(" ", event({"type": "key", "key": "a", "shift": True}))
print(" ", event({"type": "scroll"}), "|", event([1]))
''')

  [] -> empty
  ['quit'] -> bye
  ['go', 'north'] -> walk north
  ['add', '1', '2', '3'] -> sum 6
  ['a', 'b', 'c'] -> first=a last=c
  ['go', 'east'] -> first=go last=east
  ('quit',) -> bye
  'quit' -> ?
  click at 3,4
  key a extras=['shift']
  unhandled scroll | not an event


## 05. Matching - Guards, Or-Patterns, As, Constants

- Guard: case pattern if condition. Pattern must match FIRST, then the guard
  runs; if the guard is false, matching continues with the next case.
- Or-pattern A | B: every alternative must bind the SAME names.
- "pattern as name" captures the whole matched piece.
- To match a CONSTANT by name use a dotted name (Color.RED, mod.LIMIT);
  a plain name is a capture, never a comparison.

In [6]:
run((3, 10), '''
import enum

class Color(enum.Enum):
    RED = 1
    GREEN = 2

def paint(c):
    match c:
        case Color.RED:
            return "stop"
        case Color.GREEN:
            return "go"
        case _:
            return "unknown"

def bucket(n):
    match n:
        case int() | float() if n < 0:
            return "negative"
        case 0:
            return "zero"
        case int(x) | float(x) if x < 10:
            return "small"
        case _:
            return "big"

print("enum constants:", paint(Color.RED), paint(Color.GREEN), paint("x"))
print("guards:", [bucket(v) for v in (-1, 0, 5, 2.5, 50)])
match [1, [2, 3]]:
    case [a, [b, *tail] as inner]:
        print("as + nested:", a, b, tail, inner)
''')

enum constants: stop go unknown
guards: ['negative', 'zero', 'small', 'small', 'big']
as + nested: 1 2 [3] [2, 3]


## 06. Matching Gotchas

- A bare name in a case ALWAYS captures (and rebinds that variable!).
      RED = 1
      match x:  case RED: ...   # matches EVERYTHING and overwrites RED
  Python flags a capture that makes later cases unreachable
  (SyntaxError: "name capture 'RED' makes remaining patterns unreachable").
- match str is NOT a sequence: case [a, b] will not split "ab".
- 1 == 1.0 == True, so case 1 matches True and 1.0. Order and class patterns
  (int()) matter if the difference matters; bool is a subclass of int.
- Variables bound in a case stay bound after the match (no new scope).
- "match" and "case" are SOFT keywords: still fine as variable names
  (match = re.match(...) still works), which is why old code does not break.
- Not a performance trick: chain of tests is evaluated in order.

In [7]:
run((3, 10), '''
RED = 1
try:
    compile("match 5:\\n    case RED:\\n        pass\\n    case 6:\\n        pass\\n", "<demo>", "exec")
except SyntaxError as err:
    print("bare-name capture flagged:", "unreachable" in str(err))
match "ab":
    case [a, b]:
        print("str matched as sequence?!")
    case _:
        print("a str does not match a sequence pattern")
match True:
    case 1:
        print("True matches case 1 (True == 1)")
match 1.0:
    case 1:
        print("1.0 matches case 1 too")
match 7:
    case seen:
        pass
print("capture variable survives the match:", seen)
match = "match is still a usable name"
print(match)
''')

bare-name capture flagged: True
a str does not match a sequence pattern
True matches case 1 (True == 1)
1.0 matches case 1 too
capture variable survives the match: 7
match is still a usable name


## 07. Parenthesized Context Managers (3.10 / officially 3.10)

- You can wrap several "with" items in parentheses and spread them over
  lines, with a trailing comma - no backslashes needed.
- Works in 3.10 in practice (new parser), documented as 3.10 syntax.

In [8]:
run((3, 10), '''
import contextlib, io
with (
    contextlib.redirect_stdout(io.StringIO()) as captured,
    contextlib.suppress(ZeroDivisionError),
):
    print("hidden text")
    1 / 0
print("captured:", repr(captured.getvalue()), "| error suppressed, we are still running")
''')

captured: 'hidden text\n' | error suppressed, we are still running


## 08. ExceptionGroup And except* (3.11)

- Sometimes SEVERAL errors happen at once (many tasks failed, many form
  fields invalid). ExceptionGroup(message, [exc1, exc2, ...]) carries them all.
- "except* Type" handles the members of that type; each except* clause may
  run, and unhandled members are re-raised in a new group.
- asyncio.TaskGroup raises ExceptionGroup when several tasks fail.
- Use plain "except" for ONE error; "except*" only when groups are possible.

In [9]:
run((3, 11), '''
def many_failures():
    raise ExceptionGroup("batch failed", [ValueError("bad value"), TypeError("bad type"), ValueError("another value")])

try:
    many_failures()
except* ValueError as eg:
    print("ValueError group size:", len(eg.exceptions), [str(e) for e in eg.exceptions])
except* TypeError as eg:
    print("TypeError group size:", len(eg.exceptions))

try:
    try:
        raise ExceptionGroup("g", [KeyError("k"), OSError("o")])
    except* KeyError:
        print("handled KeyError, OSError left over")
except ExceptionGroup as rest:
    print("leftover:", [type(e).__name__ for e in rest.exceptions])
''')

ValueError group size: 2 ['bad value', 'another value']
TypeError group size: 1
handled KeyError, OSError left over
leftover: ['OSError']


## 09. add_note And ExceptionGroup Methods (3.11)

- exc.add_note("text") attaches extra context; shown in the traceback after
  the message and stored in exc.__notes__ (a list).
- group.split(cond) -> (matching, rest); group.subgroup(cond) -> matching part.
  cond can be an exception type, a tuple of types or a predicate function.
- except* gotchas: cannot mix "except" and "except*" in one try; cannot
  use return/break/continue inside an except* block; an ExceptionGroup
  cannot be empty or contain non-exception objects.

In [10]:
run((3, 11), '''
e = ValueError("bad")
e.add_note("while reading config")
e.add_note("line 12")
print("notes:", e.__notes__)
print("notes appear in the traceback text:", "while reading config" in "".join(traceback.format_exception(e)))
eg = ExceptionGroup("g", [ValueError(1), TypeError(2), ValueError(3)])
match_part, rest = eg.split(ValueError)
print("split:", [repr(x) for x in match_part.exceptions], "| rest:", [repr(x) for x in rest.exceptions])
print("subgroup(TypeError):", [repr(x) for x in eg.subgroup(TypeError).exceptions], "| message kept:", eg.subgroup(TypeError).message)
try:
    ExceptionGroup("empty", [])
except ValueError as err:
    print("empty group rejected:", err)
''')

notes: ['while reading config', 'line 12']
notes appear in the traceback text: True
split: ['ValueError(1)', 'ValueError(3)'] | rest: ['TypeError(2)']
subgroup(TypeError): ['TypeError(2)'] | message kept: g
empty group rejected: second argument (exceptions) must be a non-empty sequence


## 10. PEP 695 Generics: def f[T] and class C[T] (3.12)

- Before 3.12: T = TypeVar("T") at module level, then def first(x: list[T]) -> T.
- 3.12: write the type parameter right in the definition:
      def first[T](items: list[T]) -> T: ...
      class Stack[T]: ...
  No import, no separate TypeVar line, and the name is scoped to the definition.
- Bounds and constraints: [T: int] (bound) and [T: (int, str)] (constraints).
- ParamSpec/TypeVarTuple forms: [**P], [*Ts].
- Types are not checked when the program runs; they are for tools like mypy,
  pyright and your editor. The syntax is just nicer.

In [11]:
run((3, 12), '''
def first[T](items: list[T]) -> T:
    return items[0]

class Stack[T]:
    def __init__(self) -> None:
        self._items: list[T] = []
    def push(self, item: T) -> None:
        self._items.append(item)
    def pop(self) -> T:
        return self._items.pop()

def biggest[N: (int, float)](a: N, b: N) -> N:
    return a if a > b else b

s = Stack[int]()
s.push(1); s.push(2)
print("first:", first(["a", "b"]), "| stack pop:", s.pop(), "| biggest:", biggest(3, 9))
print("type params of first:", [p.__name__ for p in first.__type_params__])
print("bound/constraints stored:", biggest.__type_params__[0].__constraints__)
print("no runtime checking:", first([1.5]), "| wrong type still runs:", biggest("a", "b"))
''')

first: a | stack pop: 2 | biggest: 9
type params of first: ['T']
bound/constraints stored: (<class 'int'>, <class 'float'>)
no runtime checking: 1.5 | wrong type still runs: b


## 11. PEP 695: the type Statement (3.12)

- "type Name = <type expression>" makes a TypeAliasType object.
- The right side is evaluated LAZILY (only when you read .__value__), so
  forward and recursive aliases work with no string quotes:
      type Tree = int | list[Tree]
- It can be generic: type Pair[T] = tuple[T, T]
- Gotcha: "type" is a SOFT keyword, so a variable named type still works,
  but "type X = int" is a statement and NOT a function call.
- Before 3.12: X: TypeAlias = ... (typing.TypeAlias, 3.10).

In [12]:
run((3, 12), '''
type IntList = list[int]
type Pair[T] = tuple[T, T]
type Tree = int | list[Tree]
print("alias name/value:", IntList.__name__, IntList.__value__)
print("generic alias params:", [p.__name__ for p in Pair.__type_params__])
print("recursive alias evaluates lazily, no NameError:", Tree.__value__ is not None)
print("alias class:", type(IntList).__name__)
''')

alias name/value: IntList list[int]
generic alias params: ['T']
recursive alias evaluates lazily, no NameError: True
alias class: TypeAliasType


## 12. Typing: Self, LiteralString, Unpack, Required/NotRequired, override

- Self (3.11): "returns an instance of the same class", good for fluent
  methods and alternative constructors in subclasses.
- LiteralString (3.11): a str built only from literals; lets a type checker
  catch SQL built from user input.
- TypedDict + Required / NotRequired (3.11): mark single keys optional or
  required without making two classes.
- Unpack (3.11): *args: *Ts / **kwargs: Unpack[TD] to type keyword arguments
  from a TypedDict.
- @override (3.12): says "I mean to override a parent method"; checkers
  complain if the parent method is renamed. At runtime it just sets
  __override__ = True (3.12).
- These are hints. The helper below falls back to typing_extensions (if
  installed) and otherwise skips.

In [13]:
import typing

try:
    import typing_extensions
except ImportError:
    typing_extensions = None


def typing_name(name):
    return getattr(typing, name, None) or getattr(typing_extensions, name, None)


names = ["Self", "LiteralString", "Unpack", "Required", "NotRequired", "override", "TypeGuard", "TypeIs"]
found = {n: typing_name(n) is not None for n in names}
print("every name found in typing or typing_extensions:", all(found.values()) or sorted(n for n, ok in found.items() if not ok))
print("native in typing on this version, Self:", hasattr(typing, "Self"), "== version >= 3.11:", hasattr(typing, "Self") == (PY >= (3, 11)))
Self = typing_name("Self")
override = typing_name("override")
Required, NotRequired = typing_name("Required"), typing_name("NotRequired")
if Self and override and Required:
    class Builder:
        def __init__(self):
            self.parts = []

        def add(self, p) -> Self:
            self.parts.append(p)
            return self

    class LoudBuilder(Builder):
        @override
        def add(self, p) -> Self:
            return super().add(p.upper())

    class Movie(typing.TypedDict):
        title: Required[str]
        year: NotRequired[int]

    print("Self keeps the subclass:", type(LoudBuilder().add("a")).__name__, LoudBuilder().add("a").add("b").parts)
    print("override marker (3.12 native only):", getattr(LoudBuilder.add, "__override__", "not set"))
    print("TypedDict required/optional keys:", sorted(Movie.__required_keys__), sorted(Movie.__optional_keys__))
else:
    print("skipped: typing names unavailable (install typing_extensions)")

every name found in typing or typing_extensions: True
native in typing on this version, Self: True == version >= 3.11: True
Self keeps the subclass: LoudBuilder ['A', 'B']
override marker (3.12 native only): True
TypedDict required/optional keys: ['title'] ['year']


## 13. TypeGuard (3.10) and TypeIs (3.13)

- A user-defined "type guard" is a function returning bool that ALSO tells
  the type checker what the argument is when True.
      def is_str_list(v: list[object]) -> TypeGuard[list[str]]: ...
- TypeGuard narrows only in the True branch, and the checker must trust you.
- TypeIs (3.13) is stricter: narrows BOTH branches (True: the type; False:
  everything else), so the function must be a real isinstance-style test.
- At runtime both are ordinary functions returning bool; nothing is enforced.

In [14]:
TypeGuard = typing_name("TypeGuard")
TypeIs = typing_name("TypeIs")
if TypeGuard:
    def is_str_list(values: list) -> TypeGuard[list[str]]:
        return all(isinstance(v, str) for v in values)

    print("TypeGuard function runs as a plain bool:", is_str_list(["a", "b"]), is_str_list(["a", 1]))
if TypeIs:
    def is_int(value: object) -> TypeIs[int]:
        return isinstance(value, int)

    print("TypeIs function runs as a plain bool:", is_int(3), is_int("x"))
else:
    print("TypeIs not available (needs 3.13 or typing_extensions)")
# Gotcha: a guard that lies (returns True for the wrong type) breaks type
# safety silently; the checker believes it.

TypeGuard function runs as a plain bool: True False
TypeIs function runs as a plain bool: True False


## 14. f-string Relaxations (3.12, PEP 701)

- Before 3.12: you could not reuse the SAME quote type inside an f-string
  expression, no backslashes, no comments, and one line per expression.
- 3.12: f-strings are parsed by the real parser, so all of these work:
      f"{d["key"]}"                     same quotes inside
      f"{'\\n'.join(items)}"            backslash in an expression
      f"{ x  # comment }" on many lines multi-line expressions + comments
      f"{f"{f"{1}"}"}"                   deep nesting
- Compatibility: code using the new forms is a SyntaxError before 3.12, so
  do not use them in libraries that still support 3.11.

In [15]:
run((3, 12), '''
d = {"key": "value", "n": 3}
items = ["a", "b", "c"]
print(f"same quotes: {d["key"]}")
print(f"backslash: {"\\n".join(items)!r}")
print(f"""multi-line expression: {
    d["n"]       # a comment inside the braces
    * 2
}""")
print(f"nested: {f"{f"{1 + 1}"}"}")
print(f"format spec: {d['n']:>5}|{3.14159:.2f}|{255:#x}|{1234567:,}")
''')

same quotes: value
backslash: 'a\nb\nc'
multi-line expression: 6
nested: 2
format spec:     3|3.14|0xff|1,234,567


## 15. tomllib - Reading TOML (3.11)

- TOML is the config format used by pyproject.toml. tomllib reads it (only
  READS; there is no writer in the standard library).
- tomllib.loads(str) and tomllib.load(file opened in BINARY "rb" mode).
- Types come back as Python types: tables -> dict, arrays -> list, dates ->
  datetime objects, integers -> int.
- Before 3.11 the same API lives in the third-party "tomli" package.

In [16]:
try:
    import tomllib
except ImportError:
    try:
        import tomli as tomllib
    except ImportError:
        tomllib = None
if tomllib:
    doc = tomllib.loads("""
[project]
name = "demo"
version = "1.2.3"
tags = ["a", "b"]

[tool.demo]
debug = true
ratio = 0.5
""")
    print("project name:", doc["project"]["name"], "| tags:", doc["project"]["tags"])
    print("nested table:", doc["tool"]["demo"])
    try:
        tomllib.loads("this is = = broken")
    except tomllib.TOMLDecodeError:
        print("bad TOML raises TOMLDecodeError (a ValueError)")
    import io
    try:
        tomllib.load(io.StringIO("a = 1"))
    except TypeError:
        print("load() needs a binary file; text file raises TypeError")
else:
    print("tomllib not available (3.11+ or install tomli)")

project name: demo | tags: ['a', 'b']
nested table: {'debug': True, 'ratio': 0.5}
bad TOML raises TOMLDecodeError (a ValueError)
load() needs a binary file; text file raises TypeError


## 16. itertools.pairwise (3.10) And itertools.batched (3.12)

- pairwise(it): overlapping neighbours (a,b), (b,c), (c,d) - differences,
  trends, "did it change?".
- batched(it, n): non-overlapping chunks of size n as tuples; the last one
  may be shorter. Lazy, works on any iterable. 3.13 adds strict=True.
- Before 3.12 the usual recipe is islice in a loop.

In [17]:
print("pairwise:", list(itertools.pairwise([1, 4, 9, 16])))
print("differences:", [b - a for a, b in itertools.pairwise([1, 4, 9, 16])])
if hasattr(itertools, "batched"):
    print("batched:", list(itertools.batched(range(7), 3)))
    print("batched chunks are tuples:", type(next(iter(itertools.batched("abcde", 2)))).__name__)
    try:
        list(itertools.batched("abc", 0))
    except ValueError:
        print("n must be at least 1")
    if PY >= (3, 13):
        try:
            list(itertools.batched("abcde", 2, strict=True))
        except ValueError:
            print("strict=True raises on a short final batch (3.13)")
else:
    def batched(iterable, n):
        it = iter(iterable)
        while chunk := tuple(itertools.islice(it, n)):
            yield chunk
    print("fallback batched:", list(batched(range(7), 3)))

pairwise: [(1, 4), (4, 9), (9, 16)]
differences: [3, 5, 7]
batched: [(0, 1, 2), (3, 4, 5), (6,)]
batched chunks are tuples: tuple
n must be at least 1


## 17. The int <-> str Digit Limit (3.11, backported to 3.10.7+)

- Converting a HUGE decimal string to int (or an int to str) takes time that
  grows with the square of the length - a denial-of-service risk when the
  text comes from users.
- Default limit: 4300 digits. Exceeding it raises ValueError.
- sys.get_int_max_str_digits() / set_int_max_str_digits(n); 0 means unlimited.
  Also the env var PYTHONINTMAXSTRDIGITS and the -X int_max_str_digits flag.
- Power-of-two bases (hex, binary, octal) are NOT limited: int(s, 16) is fast.
- Only decimal conversions are affected; int arithmetic is not.

In [18]:
if hasattr(sys, "get_int_max_str_digits"):
    old = sys.get_int_max_str_digits()
    try:
        sys.set_int_max_str_digits(4300)
        big = "9" * 5000
        try:
            int(big)
        except ValueError:
            print("5000-digit string rejected at the 4300 limit")
        n = 10 ** 5000
        try:
            str(n)
        except ValueError:
            print("str() of a 5001-digit int rejected too")
        print("hex of the same number is fine:", len(hex(n)) > 4000)
        sys.set_int_max_str_digits(0)
        print("limit off (0): digits parsed:", len(str(int(big))))
    finally:
        sys.set_int_max_str_digits(old)
    print("limit restored:", sys.get_int_max_str_digits() == old)
else:
    print("skipped: this Python has no digit limit")

5000-digit string rejected at the 4300 limit
str() of a 5001-digit int rejected too
hex of the same number is fine: True
limit off (0): digits parsed: 5000
limit restored: True


## 18. Better Error Messages (3.10 - 3.13)

- 3.10: "Did you mean" for misspelt names/attributes (NameError,
  AttributeError); "'(' was never closed" and similar SyntaxErrors that point
  at the real problem; missing-comma hints.
- 3.11: fine-grained error locations (carets ^^^^ under the exact
  sub-expression that failed) in tracebacks.
- 3.12: suggests "from x import y" for missing imports and "Did you mean:
  self.name" for instance attributes.
- 3.13: colored tracebacks in a terminal; the module name shadowing hint.
- These show in traceback TEXT, not inside str(err), so tools that parse
  exception strings should not depend on them.

In [19]:
class Account:
    def __init__(self):
        self.balance = 10


try:
    Account().balanse
except AttributeError as err:
    text = "".join(traceback.format_exception_only(type(err), err))
    print("str(err) has no hint:", "Did you mean" not in str(err))
    print("traceback text hint present (3.10+):", "Did you mean" in text)
try:
    compile("x = (1, 2\ny = 3\n", "<demo>", "exec")
except SyntaxError as err:
    print("unclosed bracket message mentions it:", "never closed" in str(err) or "'(' was never closed" in str(err))
try:
    compile("print('a' 'b' x)", "<demo>", "exec")
except SyntaxError as err:
    print("SyntaxError is a clear one-liner:", type(err).__name__)

str(err) has no hint: True
traceback text hint present (3.10+): True
unclosed bracket message mentions it: True
SyntaxError is a clear one-liner: SyntaxError


## 19. sys.monitoring - Cheap Event Hooks (3.12, PEP 669)

- sys.settrace / sys.setprofile slow the program down a lot. sys.monitoring
  (3.12) lets debuggers, profilers and coverage tools subscribe to specific
  events with far less overhead, and turn off a location after it fired.
- Steps: claim a TOOL ID (0-5), register a callback per event, then
  set_events(tool_id, mask). Free it with free_tool_id when done.
- Events: PY_START, PY_RETURN, LINE, CALL, RAISE, BRANCH, JUMP ...
- A callback may return sys.monitoring.DISABLE to stop events at that spot.
- For everyday code you never call this; coverage.py and debuggers do.

In [20]:
if hasattr(sys, "monitoring"):
    mon = sys.monitoring
    TOOL = mon.PROFILER_ID if hasattr(mon, "PROFILER_ID") else 2
    started = []

    def target(a, b):
        return a + b

    def on_start(code, offset):
        if code is target.__code__:
            started.append(code.co_name)

    mon.use_tool_id(TOOL, "demo-tool")
    try:
        mon.register_callback(TOOL, mon.events.PY_START, on_start)
        mon.set_events(TOOL, mon.events.PY_START)
        target(1, 2)
        target(3, 4)
        mon.set_events(TOOL, 0)
        target(5, 6)  # not counted: events are off again
    finally:
        mon.register_callback(TOOL, mon.events.PY_START, None)
        mon.free_tool_id(TOOL)
    print("calls seen while monitoring was on:", len(started), started[:1])
    print("a few event names exist:", all(hasattr(mon.events, n) for n in ("PY_START", "PY_RETURN", "LINE", "RAISE")))
else:
    print("skipped: sys.monitoring needs 3.12")

calls seen while monitoring was on: 2 ['target']
a few event names exist: True


## 20. 3.13: Free-Threaded Build, Experimental JIT, New REPL

- FREE-THREADED ("no-GIL") build (PEP 703, experimental in 3.13): a special
  interpreter (python3.13t) without the Global Interpreter Lock, so threads
  can run Python code truly in parallel on several cores.
    * Cost: slower single-thread code (a few to 20 percent), C extensions must
      be rebuilt for it, and thread-safety bugs that the GIL used to hide
      can appear. The GIL can be turned back on with PYTHON_GIL=1.
    * Detect it at run time:
          sys._is_gil_enabled()                 (3.13+, may be missing before)
          sysconfig.get_config_var("Py_GIL_DISABLED") == 1   (build option)
- EXPERIMENTAL JIT (PEP 744, 3.13): a copy-and-patch JIT compiler, only if
  Python was BUILT with --enable-experimental-jit. Usually off, small speedups
  at best. Switch with PYTHON_JIT=1 on a build that supports it. Not for
  production in 3.13.
- NEW INTERACTIVE REPL (3.13, based on PyPy's): multi-line editing, history
  across lines, colored prompt and tracebacks, F1 help, F2 history, F3 paste
  mode, plus plain "exit" and "quit" work with no parentheses. Disable with
  PYTHON_BASIC_REPL=1.

In [21]:
import sysconfig

gil_known = hasattr(sys, "_is_gil_enabled")
gil_on = sys._is_gil_enabled() if gil_known else True
free_build = bool(sysconfig.get_config_var("Py_GIL_DISABLED"))
print("sys._is_gil_enabled exists only on 3.13+:", gil_known == (PY >= (3, 13)))
print("GIL enabled (type check only):", isinstance(gil_on, bool))
print("a build that is not free-threaded always has the GIL on:", free_build or gil_on)
print("free-threaded build flag is a bool:", isinstance(free_build, bool))
print("JIT is a build option; checking for an inspection hook:", hasattr(sys, "_jit") or "not exposed on this version")
# Tip: libraries should not assume the GIL protects their data. Use locks.

sys._is_gil_enabled exists only on 3.13+: True
GIL enabled (type check only): True
a build that is not free-threaded always has the GIL on: True
free-threaded build flag is a bool: True
JIT is a build option; checking for an inspection hook: not exposed on this version


## 21. Deprecations And Removals

- 3.10: distutils deprecated; loop= parameter removed from many asyncio APIs.
- 3.11: many unittest/asyncio aliases removed; "re" flag global-in-middle
  is an error; 'binhex' removed.
- 3.12: REMOVED distutils, asyncore, asynchat, smtpd, imp. (Use setuptools
  or packaging; asyncio; importlib.) Also unittest "assertEquals"-style
  aliases removed, ssl.wrap_socket removed.
- 3.13: PEP 594 "dead batteries" removed: aifc, audioop, cgi, cgitb, chunk,
  crypt, imghdr, mailcap, msilib, nis, nntplib, ossaudiodev, pipes, sndhdr,
  spwd, sunau, telnetlib, uu, xdrlib, lib2to3. (cgi.parse -> email/urllib.)
- Also 3.13: locale.resetlocale, some typing and pathlib bits removed.
- Check programs with:  python -W error::DeprecationWarning  or pytest -W error
  (or run under -X dev) to find use of things scheduled for removal.

In [22]:
def importable(name):
    try:
        return importlib.util.find_spec(name) is not None
    except (ImportError, ValueError):
        return False


gone_312 = ["asyncore", "asynchat", "smtpd", "imp"]
gone_313 = ["cgi", "cgitb", "crypt", "telnetlib", "pipes", "imghdr", "uu", "xdrlib"]
if PY >= (3, 12):
    print("3.12 removals really missing:", [n for n in gone_312 if importable(n)] == [])
else:
    print("3.12 removals still present on older Python:", all(importable(n) for n in gone_312))
if PY >= (3, 13):
    print("3.13 dead batteries really missing:", [n for n in gone_313 if importable(n)] == [])
else:
    print("3.13 dead batteries still present before 3.13:", all(importable(n) for n in gone_313))
print("modern replacements exist: tomllib/zoneinfo/graphlib:", all(importable(n) for n in ("zoneinfo", "graphlib")) and (importable("tomllib") or PY < (3, 11)))

3.12 removals really missing: True
3.13 dead batteries still present before 3.13: True
modern replacements exist: tomllib/zoneinfo/graphlib: True


## 22. Recap

- 3.10: match/case, X | Y types, TypeAlias/TypeGuard/ParamSpec, pairwise,
        parenthesized with, better errors.
- 3.11: except*/ExceptionGroup, add_note, tomllib, Self/Required/LiteralString,
        big speedups (faster CPython), fine-grained error carets, digit limit.
- 3.12: def f[T] / type X = ..., f-string freedom, override, batched,
        sys.monitoring, per-interpreter GIL, distutils/imp removed.
- 3.13: free-threaded + JIT (experimental), new REPL, TypeIs, dead batteries
        removed, copy.replace and warnings.deprecated (3.13).
- Write code with a guard for each: version tuple for syntax, hasattr or
  try-import for APIs, and exec() or a separate module for syntax you cannot
  even parse on older versions.

In [23]:
print("lesson finished: every feature above was gated by a version or feature check")

lesson finished: every feature above was gated by a version or feature check
